# 01 Bronze ingest
Fetches ESPN scoreboards and box scores for a window of dates, lands the raw JSON in a volume,
and loads it into Bronze Delta tables.
**Parameters:** `start_date` and `end_date` (YYYY-MM-DD) are optional. Blank means the last
`lookback_days` days ending yesterday.
**Writes:** `bronze_scoreboard`, `bronze_game_summary`

In [0]:
# --- Setup: imports and constants ---
import os, sys, json, time
from datetime import date, timedelta
from pyspark.sql import functions as F   # Spark's built-in column functions

# Make our src/ package importable (the notebook lives in notebooks/, so go up one level)
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(os.path.join(repo_root, "src"))

from nba_pipeline.api_client import fetch_scoreboard_json, fetch_game_summary_json
from nba_pipeline.storage import land_raw

# Where raw files are landed
RAW_BASE = "/Volumes/workspace/nba/raw"
print("ready")

In [0]:
# --- Parameters: create the widgets, then work out the date window ---
# Widgets are input boxes at the top of the notebook. A Workflow can fill them in too.
dbutils.widgets.text("start_date", "")      # blank = auto
dbutils.widgets.text("end_date", "")        # blank = yesterday (UTC)
dbutils.widgets.text("lookback_days", "3")  # used only when start_date is blank

start_str = dbutils.widgets.get("start_date").strip()
end_str = dbutils.widgets.get("end_date").strip()
lookback = int(dbutils.widgets.get("lookback_days"))

# Defaults: end = yesterday, start = (lookback_days - 1) days before the end
end = date.fromisoformat(end_str) if end_str else date.today() - timedelta(days=1)
start = date.fromisoformat(start_str) if start_str else end - timedelta(days=lookback - 1)
assert start <= end, f"start_date {start} is after end_date {end}"

print(f"Window: {start} to {end} ({(end - start).days + 1} days)")

In [0]:
# --- Fetch the scoreboard for every date in the window ---
# Overwritten each run because results can change; the file name is built from the date,
# so re-running never creates duplicates.
failed_days = []

d = start
while d <= end:
    try:
        payload = fetch_scoreboard_json(d)   # retries happen inside the client
        land_raw(payload, RAW_BASE, "scoreboard", f"game_date={d.isoformat()}")
    except Exception as exc:
        # One bad day shouldn't stop the others; record it and raise at the end
        print(f"{d}: FAILED: {exc}")
        failed_days.append(d.isoformat())
    time.sleep(1.5)   # be polite to the API
    d += timedelta(days=1)

print(f"scoreboards done: {(end - start).days + 1 - len(failed_days)} ok, {len(failed_days)} failed")

In [0]:
# --- Fetch box scores for final games in the window that we don't have yet ---

# 1. Read the scoreboard files we just landed and collect the IDs of FINAL games
final_ids = []
d = start
while d <= end:
    path = f"{RAW_BASE}/scoreboard/game_date={d.isoformat()}.json"
    if os.path.exists(path):   # a day that failed above has no file
        with open(path, encoding="utf-8") as f:
            data = json.load(f)
        for ev in data.get("events", []):
            if ev["status"]["type"]["name"] == "STATUS_FINAL":
                final_ids.append(ev["id"])
    d += timedelta(days=1)

# 2. Fetch each box score, skipping any already landed (final games never change)
fetched, skipped, failed_games = 0, 0, []
for game_id in final_ids:
    if os.path.exists(f"{RAW_BASE}/game_summary/game_id={game_id}.json"):
        skipped += 1
        continue
    try:
        payload = fetch_game_summary_json(game_id)
        land_raw(payload, RAW_BASE, "game_summary", f"game_id={game_id}")
        fetched += 1
    except Exception as exc:
        print(f"game {game_id}: FAILED: {exc}")
        failed_games.append(game_id)
    time.sleep(1.5)

print(f"final games in window: {len(final_ids)} | fetched: {fetched} | already had: {skipped} | failed: {len(failed_games)}")

In [0]:
# --- Fail loudly: if any fetch failed, stop BEFORE touching the Bronze tables ---
# In a Workflow this marks the task failed, so Silver and Gold don't run on incomplete
# data. The next run's lookback window retries the failed days automatically.
if failed_days or failed_games:
    raise RuntimeError(f"Ingestion failed. Days: {failed_days} | Games: {failed_games}")
print("all fetches succeeded")

In [0]:
# --- Reusable Bronze loader: raw files in a volume folder -> Delta table ---
def load_bronze(entity: str, table: str) -> int:
    """Read every raw file in RAW_BASE/<entity>/ (one row per file) and
    write it to a Delta table. Returns the table's row count."""

    # Describe the read (lazy): treat each file as one row of text
    raw_df = (
        spark.read
        .format("text")
        .option("wholetext", True)
        .load(f"{RAW_BASE}/{entity}/")
    )

    # Add the Bronze metadata columns (still lazy)
    bronze_df = raw_df.select(
        F.col("value").alias("raw_json"),
        F.col("_metadata.file_path").alias("source_file"),
        F.current_timestamp().alias("ingested_at"),
    )

    # The ACTION: read the files and write the Delta table (overwrite = re-runnable)
    bronze_df.write.format("delta").mode("overwrite").saveAsTable(table)

    # Read the table back and return its row count as a sanity check
    return spark.table(table).count()


# Load both Bronze tables from everything in the volume (the full history, not just the window)
n_scores = load_bronze("scoreboard", "workspace.nba.bronze_scoreboard")
n_boxes = load_bronze("game_summary", "workspace.nba.bronze_game_summary")
print(f"bronze_scoreboard: {n_scores} rows | bronze_game_summary: {n_boxes} rows")